In [1]:
import os
import pandas as pd

from lib.query import Query

In [ ]:
RESULTS = {
    'QWEN_QWENReason': '../02_data/2026-06-08_compendium_judged/qwen_qwen-reasoning',
    'QWENReason_QWEN': '../02_data/2026-06-09_compendium_judged/qwen-reasoning_qwen',
    'QWENReason_GPT': '../02_data/2026-06-08_compendium_judged/qwen-reasoning_gpt',
    'GPT_QWEN': '../02_data/2026-06-04_compendium_qwen-judged/openai',
    'GPT_GPT': '../02_data/2026-06-04_compendium/openai',
    'GPT_QWENReason': '../02_data/2026-06-08_compendium_judged/gpt_qwen-reasoning',
    'QWEN_QWEN': '../02_data/2026-06-02_compendium/qwen',
    'QWEN_GPT': '../02_data/2026-06-07_compendium_gpt-judged/qwen',
    'QWENReason_QWENReason': '../02_data/2026-06-08_compendium_judged/qwenres_qwenres',
}

MODELS = {
    'QWEN': 'Qwen3.5-9B',
    'QWENReason': 'Qwen3.5-9B Reasoning',
    'GPT': 'gpt-5.4-mini-2026-03-17'
}

In [ ]:
sparse = '../01_config/2026-05-26/compendium/generated/qwen/sparse.jsonl'
dense = '../01_config/2026-05-26/compendium/generated/qwen/dense.jsonl'
hybrid = '../01_config/2026-05-26/compendium/generated/qwen/hybrid.jsonl'

In [4]:
def recall_at_k(query: Query, k: int = 10) -> float:
    retrieved = [p.global_id for p in query.retrieved]
    references = set([p.global_id for p in query.reference])

    top_k = retrieved[:k]
    hits = sum(1 for doc in top_k if doc in references)
    return hits / len(references) if references else 0.0

In [6]:
df_sparse = pd.read_json(sparse, lines=True, orient='records')
df_sparse['id'] = df_sparse['id'].astype(str)

df_dense = pd.read_json(dense, lines=True, orient='records')
df_dense['id'] = df_dense['id'].astype(str)

df_hybrid = pd.read_json(hybrid, lines=True, orient='records')
df_hybrid['id'] = df_hybrid['id'].astype(str)

retrievers = {
    'sparse': df_sparse,
    'dense': df_dense,
    'hybrid': df_hybrid
}

In [8]:
results = {}

for exp in RESULTS.keys():
    setup = exp.split('_')
    path = RESULTS[exp]

    generator = setup[0]
    judge = setup[1]

    for id in os.listdir(path):
        if id == '.DS_Store':
            continue
        info = id.split('_')
        retriever = info[3]

        try:
            df = pd.read_json(f"{path}/{id}/results.jsonl", orient='records', lines=True)
        except:
            df = pd.read_json(f"{path}/{id}/results.json", orient='records')

        df['id'] = df['id'].astype(str)
        df.update(retrievers[retriever][['reference']])
        df['retriever_recall@10'] = df.apply(lambda r: recall_at_k(Query.model_validate(r.to_dict()), 10), axis=1)

        df['query-id'] = df['id']
        df['retriever'] = retriever
        df['generator'] = MODELS[generator]
        df['judge'] = MODELS[judge]


        df['recall'] = df['retriever_recall@10']
        df['task_success'] = df['task_success_score']

        results[f"{retriever}_{generator}_{judge}"] = df[['query-id', 'retriever', 'generator', 'judge', 'recall', 'task_success']]
        # break

/var/folders/d_/r89f0qt56b91xr3wj5bwft480000gr/T/ipykernel_29666/2524374315.py:17: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(f"{path}/{id}/results.jsonl", orient='records', lines=True)
/var/folders/d_/r89f0qt56b91xr3wj5bwft480000gr/T/ipykernel_29666/2524374315.py:17: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(f"{path}/{id}/results.jsonl", orient='records', lines=True)
/var/folders/d_/r89f0qt56b91xr3wj5bwft480000gr/T/ipykernel_29666/2524374315.py:17: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(f"{path}/{id}/results.jsonl", orient='records', lines=True)
/var

In [9]:
df = pd.concat(results).reset_index(drop=True)
df

,query-id,retriever,generator,judge,recall,task_success
0,1,dense,Qwen3.5-9B,Qwen3.5-9B Reasoning,0.040000,3
1,2,dense,Qwen3.5-9B,Qwen3.5-9B Reasoning,0.500000,6
2,3,dense,Qwen3.5-9B,Qwen3.5-9B Reasoning,0.500000,3
3,4,dense,Qwen3.5-9B,Qwen3.5-9B Reasoning,0.500000,10
4,5,dense,Qwen3.5-9B,Qwen3.5-9B Reasoning,0.571429,5
...,...,...,...,...,...,...
16141,594,hybrid,Qwen3.5-9B Reasoning,Qwen3.5-9B Reasoning,0.666667,10
16142,595,hybrid,Qwen3.5-9B Reasoning,Qwen3.5-9B Reasoning,0.571429,6
16143,596,hybrid,Qwen3.5-9B Reasoning,Qwen3.5-9B Reasoning,0.800000,7
16144,597,hybrid,Qwen3.5-9B Reasoning,Qwen3.5-9B Reasoning,0.500000,10


In [11]:
df.groupby(['generator', 'judge', 'retriever']).agg(
    count=('recall', 'size'),
    recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'),
    task_success_mean=('task_success', 'mean'),
    task_success_std=('task_success', 'std')
).reset_index()

,generator,judge,retriever,count,recall_mean,recall_std,task_success_mean,task_success_std
0,Qwen3.5-9B,Qwen3.5-9B,dense,598,0.520102,0.329697,4.757525,2.438469
1,Qwen3.5-9B,Qwen3.5-9B,hybrid,598,0.496273,0.328887,4.563545,2.444385
2,Qwen3.5-9B,Qwen3.5-9B,sparse,598,0.380973,0.326993,4.132107,2.658246
3,Qwen3.5-9B,Qwen3.5-9B Reasoning,dense,598,0.520102,0.329697,5.770903,2.466395
4,Qwen3.5-9B,Qwen3.5-9B Reasoning,hybrid,598,0.496273,0.328887,5.623746,2.390077
5,Qwen3.5-9B,Qwen3.5-9B Reasoning,sparse,598,0.380973,0.326993,4.974916,2.730381
6,Qwen3.5-9B,gpt-5.4-mini-2026-03-17,dense,598,0.520102,0.329697,4.013378,2.191767
7,Qwen3.5-9B,gpt-5.4-mini-2026-03-17,hybrid,598,0.496273,0.328887,3.765886,2.186154
8,Qwen3.5-9B,gpt-5.4-mini-2026-03-17,sparse,598,0.380973,0.326993,3.215719,2.237630
9,Qwen3.5-9B Reasoning,Qwen3.5-9B,dense,598,0.520102,0.329697,5.381271,2.563456
